# 1D charge-density comparison

This Julia Jupyter notebook compares every `site_density.csv` found below a downloaded campaign result directory. It also reports particle number, final energy, residuals, and bulk staggered density order.

It is analysis-only: it neither activates nor modifies the solver environment.

## Local setup

After downloading the campaign result directory from the cluster, create a separate local plotting environment:

```bash
ANALYSIS_ENV="$HOME/julia-analysis-envs/mpo_density"
mkdir -p "$ANALYSIS_ENV"
julia --project="$ANALYSIS_ENV" -e 'using Pkg; Pkg.add(["IJulia", "CairoMakie"])'
julia --project="$ANALYSIS_ENV" -e "using IJulia; installkernel(\"Julia MPO density\"; env=Dict(\"JULIA_PROJECT\" => \"$ANALYSIS_ENV\"))"
```

Then open this notebook with your usual Jupyter or VS Code interface, select the `Julia MPO density` kernel, and set `result_root` in the next code cell.

In [ ]:
using CairoMakie
using Statistics
using TOML

In [ ]:
# Change this to the downloaded cluster result directory.
result_root = raw"/CHANGE/THIS/TO/aubry_andre_nn_l10_seed0p1"

In [ ]:
unquote(value::AbstractString) = strip(strip(value), '"')

function read_site_density(path::AbstractString)
    lines = readlines(path)
    isempty(lines) && error("empty density CSV: $path")
    strip(lines[1]) == "\"site\",\"density\"" || error("unexpected header in $path")
    sites, density = Int[], Float64[]
    for line in Iterators.drop(lines, 1)
        isempty(strip(line)) && continue
        columns = split(strip(line), ','; limit=2)
        length(columns) == 2 || error("malformed density row: $line")
        push!(sites, parse(Int, unquote(columns[1])))
        push!(density, parse(Float64, unquote(columns[2])))
    end
    sites == collect(1:length(sites)) || error("site indices are not contiguous in $path")
    return sites, density
end

function bulk_staggered_order(sites, density; edge_fraction=0.1)
    N = length(sites)
    bulk = (floor(Int, edge_fraction * N) + 1):(N - floor(Int, edge_fraction * N))
    mean((isodd(sites[i]) ? 1.0 : -1.0) * (density[i] - 0.5) for i in bulk)
end

function load_campaign(root::AbstractString)
    isdir(root) || error("result_root is not a directory: $root")
    task_dirs = sort(filter(path -> isdir(path) && isfile(joinpath(path, "site_density.csv")), readdir(root; join=true)))
    isempty(task_dirs) && error("no task directories with site_density.csv below $root")
    map(task_dirs) do directory
        sites, density = read_site_density(joinpath(directory, "site_density.csv"))
        obs = TOML.parsefile(joinpath(directory, "observables.toml"))
        (label=replace(basename(directory), r"^task_\d+_" => ""), sites=sites, density=density, observables=obs, bulk_staggered_order=bulk_staggered_order(sites, density))
    end
end

In [ ]:
cases = load_campaign(result_root)

summary = [(
    case=entry.label,
    N=length(entry.sites),
    particle_number=get(entry.observables, "particle_number", missing),
    energy_total=get(entry.observables, "energy_total", missing),
    idempotency=get(entry.observables, "idempotency_residual", missing),
    stationarity=get(entry.observables, "stationarity_residual", missing),
    bulk_staggered_order=entry.bulk_staggered_order,
) for entry in cases]

summary

## Full density and central-window view

The upper panel exposes boundary response. The lower panel magnifies a central window: persistent even/odd splitting there is a bulk Hartree--Fock CDW branch, while splitting that decays toward the centre is boundary Friedel response.

In [ ]:
CairoMakie.activate!()
figure = Figure(size=(1100, 760))
full_axis = Axis(figure[1, 1], xlabel="site", ylabel="charge density", title="Full 1D density profile")
center_axis = Axis(figure[2, 1], xlabel="site", ylabel="charge density", title="Central density window")

for entry in cases
    lines!(full_axis, entry.sites, entry.density; label=entry.label, linewidth=1.8)
    center = cld(length(entry.sites), 2)
    half_width = min(80, center - 1, length(entry.sites) - center)
    window = (center - half_width):(center + half_width)
    lines!(center_axis, entry.sites[window], entry.density[window]; label=entry.label, linewidth=1.8)
end

axislegend(full_axis; position=:rb)
axislegend(center_axis; position=:rb)
figure

In [ ]:
# Optional: write a PNG beside the downloaded results.
# save(joinpath(result_root, "charge_density_comparison.png"), figure)